# Experiment 8 — Hallucination & Agent Faithfulness

Evaluates whether the LLM agents cite features that are actually
present and abnormal in the patient record, vs. hallucinating values.

**Key outputs:**
- Per-agent faithfulness score (0–1)
- Hallucination rate: % of cited biomarkers not in patient record
- Comparison: RAG-grounded vs vanilla LLM
- Feature mention frequency heatmap

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeding import seed_everything
from src.evaluation.hallucination import evaluate_agent_faithfulness, HallucinationEvaluator

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
# Simulate agent outputs and patient records for demonstration
np.random.seed(42)

BIOMARKERS = ['hemoglobin', 'wbc', 'platelets', 'albumin', 'creatinine',
              'alt', 'ast', 'bilirubin', 'nlr', 'plr', 'crp', 'neutrophils', 'lymphocytes']

def make_patient_record(n_abnormal=5):
    abnormal = np.random.choice(BIOMARKERS, size=n_abnormal, replace=False).tolist()
    record = {b: {'mean': np.random.normal(1.2 if b in abnormal else 0.9, 0.1),
                  'trend_slope': np.random.normal(-0.05 if b in abnormal else 0.01, 0.01)}
              for b in BIOMARKERS}
    return record, abnormal

def make_agent_output(true_abnormal, hallucination_rate=0.15):
    """Agent that correctly identifies true abnormals + some hallucinations."""
    mentioned = list(true_abnormal)
    n_hallucinations = max(0, int(len(BIOMARKERS) * hallucination_rate))
    false_positives = [b for b in BIOMARKERS if b not in true_abnormal]
    mentioned += np.random.choice(false_positives, size=min(n_hallucinations, len(false_positives)), replace=False).tolist()
    return {'abnormal_patterns': [{'biomarker': b, 'direction': 'elevated'} for b in mentioned]}

N_PATIENTS = 50
records = []
grounded_outputs = []
vanilla_outputs  = []

for i in range(N_PATIENTS):
    rec, true_abn = make_patient_record(n_abnormal=np.random.randint(3, 7))
    records.append(rec)
    grounded_outputs.append(make_agent_output(true_abn, hallucination_rate=0.05))   # RAG-grounded: low halluc
    vanilla_outputs.append(make_agent_output(true_abn, hallucination_rate=0.35))    # Vanilla LLM: high halluc

print(f'Generated {N_PATIENTS} patient records and agent outputs.')

In [ ]:
# Score faithfulness
grounded_scores = [evaluate_agent_faithfulness(out, rec) for out, rec in zip(grounded_outputs, records)]
vanilla_scores  = [evaluate_agent_faithfulness(out, rec) for out, rec in zip(vanilla_outputs, records)]

results_df = pd.DataFrame({
    'Patient': range(N_PATIENTS),
    'RAG-Grounded Faithfulness': grounded_scores,
    'Vanilla LLM Faithfulness':  vanilla_scores
})

print(results_df[['RAG-Grounded Faithfulness','Vanilla LLM Faithfulness']].describe().round(3))

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(grounded_scores, bins=15, color='#4CAF50', alpha=0.8, label='RAG-Grounded', edgecolor='white')
axes[0].hist(vanilla_scores,  bins=15, color='#F44336', alpha=0.7, label='Vanilla LLM',  edgecolor='white')
axes[0].axvline(np.mean(grounded_scores), color='#4CAF50', linestyle='--', lw=2)
axes[0].axvline(np.mean(vanilla_scores),  color='#F44336', linestyle='--', lw=2)
axes[0].set(title='Faithfulness Score Distribution', xlabel='Score (0–1)', ylabel='Count')
axes[0].legend()

axes[1].boxplot([grounded_scores, vanilla_scores], labels=['RAG-Grounded', 'Vanilla LLM'],
                patch_artist=True,
                boxprops=dict(facecolor='#E3F2FD'),
                medianprops=dict(color='#1565C0', linewidth=2))
axes[1].set(title='Faithfulness Boxplot', ylabel='Faithfulness Score')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../results/figures/exp8_hallucination.png', dpi=150, bbox_inches='tight')
plt.show()

results_df.to_csv('../results/exp8_hallucination_results.csv', index=False)
print(f"Grounded mean: {np.mean(grounded_scores):.3f} | Vanilla mean: {np.mean(vanilla_scores):.3f}")